# Recruiter Dashboard AI Workflow — Developer Handoff

## One-page plain-English explanation

> *"If I throw away the existing frontend and build my own Recruiter Dashboard, exactly what API do I call, what do I send, what does the AI receive, what does it return, what gets stored, and what do I need to display?"*

**You call `POST /api/chat`** with a JSON body `{"session_id": null, "message": "Data Analyst"}`. `session_id` is `null` only on the very first message of a new job — the backend mints a UUID and hands it back to you in the response; send that exact value on every later call for this job.

**The response is not a single JSON object.** It's a **Server-Sent Events (SSE) stream** over that one POST request: zero or more `{"type": "status", "label": "..."}` progress frames, then exactly one `{"type": "result", ...}` frame, then the connection closes. Strip the `"type"` key from that final frame and you have the complete `ChatResponse` object.

**Behind that endpoint**, a LangGraph pipeline runs one Mistral LLM call per turn — structured output forced into a `TurnAnalysis` schema that, in a single shot, classifies intent, extracts job facts (title, skills, experience, ...), and writes both the next chat reply *and* quick-reply chip suggestions. Extracted facts merge into an in-memory `job_state` dict that gets (a) checkpointed by LangGraph into its own SQLite file, and (b) mirrored into a `jobs` SQL table row for cheap list/detail queries.

**The AI never writes the actual job description during chat.** That happens only via a **separate, direct** `POST /api/chat/{session_id}/generate` call — a real button click, not a side effect of any message. Publishing works the same way via `POST /api/chat/{session_id}/publish`. Both are ordinary JSON request/response (not SSE), and both return the identical `ChatResponse` shape so your UI updates the same way regardless of which endpoint you called.

**What you display, all from one `ChatResponse` object:** `messages` (transcript), `job_state` (every field collected), `jd_versions` / `selected_version` (the generated description once one exists), `phase` + `completeness_pct` (status), `suggested_options` / `asking_about_field` (chips + Skip-button eligibility). Full field-by-field guide: **Part 10** below.

**What gets stored, where:** two *separate* SQLite databases — one is LangGraph's own checkpoint store (authoritative, in-progress conversation state, keyed by `session_id`), the other is a plain `jobs` SQL table (a write-through mirror, used only for dashboard queries that never need to touch LangGraph state). See **Part 6**.

This notebook was built by inspecting the actual running codebase — every code block below is either copied verbatim from the real source (labeled **EXISTING CODE**) or is genuinely executed against it (labeled **RUN**). Nothing here is invented.

---
# Part 1 — System Overview & Inventory

## What this component does

The **Recruiter Dashboard AI workflow** lets a recruiter describe a job opening through a natural chat conversation with an AI assistant ("Arclent"). The assistant asks one question at a time, extracts structured facts from free-text answers, tracks completeness, and — only on explicit button clicks — generates a full job description and publishes it to a public job listing.

## Backend inventory (verified from source)

| Item | Actual implementation |
|---|---|
| Web framework | **FastAPI** (`backend/main.py`) |
| API routers | `backend/routes/{auth,chat,company,jobs,admin}.py`, each an `APIRouter` included into the main `app` |
| Request/response validation | **Pydantic v2** models (`backend/schemas.py`, `backend/models.py`) |
| Auth/session handling | Opaque DB-backed session tokens in an `httponly` cookie (`arclent_session`) — **not** JWT. `backend/auth.py` + `auth_sessions` table |
| Orchestration | **LangGraph** `StateGraph` (`backend/agent/graph.py`) — nodes: `load_context → analyze_turn → apply_updates → {generate_jd \| refine_jd \| publish_job \| publish_edit \| END}` |
| LLM | **Mistral** (`mistral-large-latest` by default), via `langchain_mistralai.ChatMistralAI`, called through `.with_structured_output(...)` — `backend/agent/llm.py` |
| Agent/state structure | `GraphState` — a `TypedDict` (`backend/agent/state.py`), checkpointed per-session by LangGraph |
| Database | **SQLite**, accessed via the stdlib `sqlite3` module directly (no ORM) — `backend/database.py` |
| Database files | TWO separate `.sqlite`/`.db` files: `recruitment.db` (business data) and `checkpoints.sqlite` (LangGraph's own `SqliteSaver`) |
| Existing CRUD | `backend/database.py` — `upsert_job_draft`, `save_jd_versions`, `finalize_publish`, `finalize_edit`, `list_jobs_for_company`, `list_published_jobs`, etc. |

## AI/Agent inventory

| Item | Actual implementation |
|---|---|
| Model | Mistral, `ChatMistralAI(model=MISTRAL_MODEL, api_key=MISTRAL_API_KEY, temperature=0.2)` |
| Calls per conversational turn | **Exactly one** — `call_structured(TurnAnalysis, messages, retries=2)` in `analyze_turn` |
| Calls for JD generation | A **separate**, later call — `call_structured(JobDescriptionDraft, messages, retries=2)` in `generate_jd`, only reachable via the direct `/generate` endpoint |
| Structured output enforcement | `ChatMistralAI.with_structured_output(schema)` — the model is constrained to emit exactly the Pydantic schema's shape |
| System prompt | `SYSTEM_PROMPT_TEMPLATE` in `backend/agent/prompts.py` (~200 lines) — rebuilt fresh every turn from current state |
| Conversation history | Full `state["messages"]` (every prior `HumanMessage`/`AIMessage`) is passed on every call — no truncation/summarization logic exists |

## Frontend inventory (existing implementation — reference only, per your instructions)

| Item | Actual implementation |
|---|---|
| Chat UI | `frontend/js/job-modal.js` — a vanilla-JS IIFE driving a modal on `recruiter.html` |
| HTTP calls | `frontend/js/api.js` — hand-rolled `fetch()` wrappers, including a custom SSE stream reader (no `EventSource`, no framework) |
| Endpoint used per turn | `POST /api/chat` (streamed) for conversation; `PATCH .../job-state`, `POST .../generate`, `POST .../publish`, `POST .../skip-field` for direct actions |
| State management | None beyond the DOM + one `currentData` variable holding the latest `ChatResponse` — no Redux/Vuex/etc. |

**This existing frontend is a reference implementation only.** Everything you need to build your own is the API contract documented in Parts 3–10 below.

In [1]:
# RUN — sanity check: import the extracted contract modules and confirm they're self-contained
# (no dependency on the rest of the original backend package).
import sys
sys.path.insert(0, "..")  # repo root, so `handoff` is importable

from handoff.models import TurnAnalysis, JobDescriptionDraft, JobState, Intent
from handoff.schemas import ChatRequest, ChatResponse, ChatMessage, JobStatePatch
from handoff.agent_state import GraphState
from handoff import database as handoff_db

print("Imported cleanly:")
print(" - handoff.models:", [TurnAnalysis.__name__, JobDescriptionDraft.__name__, JobState.__name__, Intent.__name__])
print(" - handoff.schemas:", [ChatRequest.__name__, ChatResponse.__name__, ChatMessage.__name__, JobStatePatch.__name__])
print(" - handoff.agent_state.GraphState fields:", list(GraphState.__annotations__.keys()))


Imported cleanly:
 - handoff.models: ['TurnAnalysis', 'JobDescriptionDraft', 'JobState', 'Intent']
 - handoff.schemas: ['ChatRequest', 'ChatResponse', 'ChatMessage', 'JobStatePatch']
 - handoff.agent_state.GraphState fields: ['messages', 'company_profile', 'recruiter_name', 'job_state', 'phase', 'job_id', 'jd_versions', 'selected_version', 'jd_stale', 'missing_essential', 'last_response', 'pending_analysis', 'last_intent', 'jd_needs_refresh', 'asking_about_field', 'suggested_options', 'options_multi_select', 'skills_followup_count', 'skipped_checklist_fields']


---
# Part 2 — Architecture Diagram

```text
 New Frontend (yours — anything that speaks HTTP + SSE)
        │  POST /api/chat  { "session_id": null|"<uuid>", "message": "Data Analyst" }
        ▼
 FastAPI route  (backend/routes/chat.py :: post_chat)
        │  authorizes session, wraps HumanMessage, opens a StreamingResponse
        ▼
 LangGraph StateGraph.stream(...)   (backend/agent/graph.py)
        │
        ├─▶ load_context     hydrate company_profile / recruiter_name (turn 1 only)
        │
        ├─▶ analyze_turn     ONE Mistral call, structured output = TurnAnalysis
        │        │
        │        ▼
        │   AI MODEL (Mistral, mistral-large-latest)
        │   in:  system prompt (company + job_state + phase) + full message history
        │   out: TurnAnalysis  (intent, field_updates, response, suggested_options, ...)
        │
        ├─▶ apply_updates    merges TurnAnalysis into job_state, computes phase/sufficiency,
        │                    WRITE-THROUGHS to SQL (jobs table) if job_title is known
        │
        └─▶ route_after_apply → refine_jd | END   (generate_jd / publish_job are NEVER
                                                     auto-routed — direct-endpoint only)
        │
        ▼
 GraphState is checkpointed  (LangGraph SqliteSaver → checkpoints.sqlite)
        │
        ▼
 ChatResponse built  (backend/routes/chat.py :: _to_response)
        │  fresh SQL read of the `jobs` row folded in alongside the LangGraph state
        ▼
 SSE "result" event  ──▶  New Frontend
        │
        ▼
 Recruiter Dashboard / Draft — rendered ENTIRELY from this one response object
```

Two things this diagram makes explicit that are easy to miss:
1. **Two persistence layers**, written at different points (LangGraph checkpoint automatically after every node; `jobs` SQL row explicitly inside `apply_updates`).
2. **`generate_jd` and `publish_job` are graph nodes that exist, but `route_after_apply` never routes to them from a normal chat turn** — they're only ever invoked directly by their own dedicated endpoints. This is a deliberate design decision, not an oversight (see Part 4).

---
# Part 3 — Frontend → Backend Request

## `POST /api/chat`

| | |
|---|---|
| **Method** | `POST` |
| **Headers** | `Content-Type: application/json`; auth via the `arclent_session` cookie (not a header you set manually — the browser sends it automatically once signed in) |
| **Request body** | `ChatRequest` (see below) |
| **Response** | `text/event-stream` (SSE) — see Part 5 |

```python
class ChatRequest(BaseModel):
    session_id: str | None = None   # null on the first message of a new job
    message: str                    # exactly what the recruiter typed (or a chip's label)
```

### Example request

In [2]:
import json

example_first_turn = {"session_id": None, "message": "Data Analyst"}
example_later_turn = {"session_id": "3f1a9c2e-...", "message": "2-3 years"}

print(json.dumps(example_first_turn, indent=2))
print(json.dumps(example_later_turn, indent=2))


{
  "session_id": null,
  "message": "Data Analyst"
}
{
  "session_id": "3f1a9c2e-...",
  "message": "2-3 years"
}


There is also `POST /api/chat/upload` (multipart form: `session_id`, `message`, `file`) for attaching a JD document — same SSE response shape, out of scope for the core contract but mentioned for completeness. **Cannot determine from the current code** whether file size/type limits are enforced anywhere beyond what `document_extract.py` itself raises on.

---
# Part 4 — Backend Request Handling

**EXISTING CODE** — `backend/routes/chat.py`, reproduced verbatim (not copied into `handoff/` as a standalone module, since it depends on the live LangGraph graph object and FastAPI's `Depends` auth injection — see the README's "what is NOT in this folder" note):

```python
def _resolve_and_authorize_session(session_id: str | None, user: dict) -> str:
    # Establishes/verifies ownership BEFORE any graph turn runs.
    if session_id is None:
        session_id = str(uuid.uuid4())
        create_chat_session(session_id, user["id"], user["company_id"])
        return session_id
    owner = get_chat_session_owner(session_id)
    if owner is None:
        raise HTTPException(status_code=404, detail="No conversation found for this session_id")
    if owner["company_id"] != user["company_id"]:
        raise HTTPException(status_code=403, detail="You don't have access to this conversation.")
    return session_id


@router.post("")
def post_chat(body: ChatRequest, user: dict = Depends(get_current_recruiter)) -> StreamingResponse:
    session_id = _resolve_and_authorize_session(body.session_id, user)
    return StreamingResponse(
        stream_graph_turn(session_id, HumanMessage(content=body.message),
                           "Understanding your request...", user),
        media_type="text/event-stream",
    )
```

`get_current_recruiter` is a FastAPI dependency (`backend/auth.py`) that reads the `arclent_session` cookie, looks up the session in the `auth_sessions` table, and returns the user dict — this is where `company_id`/`user_id` originate for everything downstream. **INTEGRATION NOTE**: your platform will almost certainly swap this for its own auth dependency; everything downstream only needs it to yield an object with `id` and `company_id`.

`stream_graph_turn` (also in `chat.py`) is where the actual LangGraph run happens:

```python
def stream_graph_turn(session_id, human_message, initial_label, user):
    graph = get_compiled_graph()
    config = {"configurable": {"thread_id": session_id,
                                "company_id": user["company_id"], "user_id": user["id"]}}
    yield _sse("status", {"label": initial_label})
    for update in graph.stream({"messages": [human_message]}, config, stream_mode="updates"):
        for node_name, node_output in update.items():
            ...  # emits honest status labels as real nodes actually start
    final_state = graph.get_state(config).values
    response = _to_response(session_id, final_state)
    yield _sse("result", response.model_dump())
```

Note `thread_id: session_id` — this is the LangGraph checkpoint key. Everything the graph needs to remember about this conversation is looked up by that one string.

## Complete endpoint reference

Every endpoint relevant to this component, in the exact template requested. All except the two
public ones require the `arclent_session` cookie (i.e. `Depends(get_current_recruiter)`). None of
them accept any header beyond `Content-Type: application/json` (or nothing, for the no-body
endpoints) — auth is cookie-based, not header-based.

---

### `POST /api/chat` — start or continue a conversational turn
- **Headers**: `Content-Type: application/json`
- **Request body**: `{"session_id": str | null, "message": str}`
- **Response**: `text/event-stream` (SSE) — see Part 9. Terminal frame is the full `ChatResponse`.
- **Backend processing**: mints/validates `session_id` → runs the full LangGraph turn (one Mistral call) → write-through to `jobs` SQL if `job_title` known → checkpoints `GraphState`.

### `POST /api/chat/upload` — same, with an attached document
- **Headers**: multipart form (`Content-Type: multipart/form-data`, set automatically by `FormData`)
- **Request body** (form fields): `session_id` (optional), `message` (optional), `file` (required)
- **Response**: identical SSE shape to `POST /api/chat`
- **Backend processing**: extracts text from the file (`document_extract.py`), prepends it to the message as `[Uploaded document: ...]`, then runs the identical graph turn — the model's `DOCUMENT_REVIEW` intent handles the rest.

### `GET /api/chat/{session_id}` — reopen an existing conversation
- **Headers**: none beyond the auth cookie
- **Request body**: none
- **Response**: plain JSON `ChatResponse` (NOT streamed)
- **Backend processing**: reads the LangGraph checkpoint if one exists; otherwise, if the job is already published, synthesizes a read-only view directly from the `jobs` SQL row (no graph run, no LLM call). See Part 12.

### `PATCH /api/chat/{session_id}/job-state` — direct, silent field edit
- **Headers**: `Content-Type: application/json`
- **Request body**: `JobStatePatch` — `{"field_updates": {}, "list_operations": [], "company_overrides": {}, "jd_text_updates": {}}` (all optional/default-empty)
- **Example**: `{"field_updates": {"experience": "4-6 years"}}`
- **Response**: plain JSON `ChatResponse`
- **Backend processing**: calls the SAME `apply_field_changes` helper `apply_updates` uses, but with **no LLM call and no chat message added**. Write-throughs to SQL exactly like a normal turn would.

### `POST /api/chat/{session_id}/generate` — write the job description (button-only)
- **Headers**: none required beyond auth
- **Request body**: none
- **Response**: plain JSON `ChatResponse` (now including `jd_versions`); **400** if the hard floor (title + skills-or-responsibilities) isn't met
- **Backend processing**: calls the `generate_jd` graph node function directly (one Mistral call, `JobDescriptionDraft` schema) — bypasses `analyze_turn`/the graph's routing entirely. This is the ONLY code path that ever writes JD content.

### `POST /api/chat/{session_id}/skip-field` — decline the current optional question
- **Headers**: none required beyond auth
- **Request body**: none
- **Response**: plain JSON `ChatResponse`
- **Backend processing**: deterministically picks the next unresolved standard-checklist field (no LLM call at all) using the SAME logic the skills-loop cap uses; remembers the skip in `skipped_checklist_fields` so it's never re-asked. Adds no user message to the transcript — only a new assistant message.

### `POST /api/chat/{session_id}/publish` — make the job (or an edit) live
- **Headers**: none required beyond auth
- **Request body**: none
- **Response**: plain JSON `ChatResponse`; **400** if no JD exists, it's stale, or the job is already published and not mid-edit
- **Backend processing**: calls `publish_job` (new job — allocates a Job ID) or `publish_edit` (already-published job — updates content in place, no new Job ID) directly, same "no graph routing" pattern as `/generate`.

### `GET /api/jobs` — this recruiter's own job list
- **Headers**: none beyond auth
- **Response**: JSON array of job rows (draft + published), this company only — see Part 13.

### `GET /api/public/jobs` / `GET /api/public/jobs/{job_id}` — public listing
- **Headers**: none — **no authentication required**
- **Response**: JSON array (list) / single JSON object (detail, includes full `selected_jd`) — published jobs, all companies.

---
# Part 5 — Model Input

The model is invoked once per turn inside `analyze_turn` (`backend/agent/nodes.py`):

```python
def analyze_turn(state: GraphState) -> dict:
    system_prompt = build_system_prompt(
        company_profile=state.get("company_profile", {}),
        job_state=state.get("job_state", {}),
        phase=state.get("phase", "collecting"),
        missing_essential=state.get("missing_essential", []),
        jd_exists=bool(state.get("jd_versions")),
        jd_stale=bool(state.get("jd_stale", False)),
        recruiter_name=state.get("recruiter_name"),
    )
    messages = [SystemMessage(content=system_prompt), *state["messages"]]
    analysis = call_structured(TurnAnalysis, messages, retries=2)
```

The model receives **two things**: one `SystemMessage` (freshly rebuilt every single turn) and the **entire prior conversation** as a list of `HumanMessage`/`AIMessage` objects.

## Model input — field by field

| Field | Type | Purpose | Where it comes from | Reaches backend via | Every turn, or once? |
|---|---|---|---|---|---|
| `company_profile` | `dict` | Reusable company context (overview, culture, benefits...) woven into the prompt so the model never invents facts | `company_profile` SQL table | Hydrated by `load_context` from `config["configurable"]["company_id"]` (itself from the auth'd user) | Fetched once (turn 1), then cached in checkpointed `GraphState` |
| `job_state` | `dict` | Everything already known about this job — the model is told never to re-ask for these | Accumulated in-graph, from prior turns' `field_updates`/`list_operations` | Read from checkpointed `GraphState` | Every turn (reflects all prior turns) |
| `phase` | `str` | One of `collecting/summary/jd_selection/publish_confirm/published/editing` — changes prompt behavior (e.g. "published" tells the model this job is already live) | Computed by `apply_updates` from the previous turn | `GraphState` | Every turn |
| `missing_essential` | `list[str]` | Fields still blocking the hard floor (job_title + skills-or-responsibilities) | Computed by `apply_updates`/`sufficiency.py` | `GraphState` | Every turn |
| `jd_exists` / `jd_stale` | `bool` | Whether a JD has been generated, and whether it's now out of date | Computed from `state["jd_versions"]` / `state["jd_stale"]` | `GraphState` | Every turn |
| `recruiter_name` | `str \| None` | First name only, for occasional natural personalization ("Nice choice, Umer!") | `users` table, via `user_id` in the auth context | Hydrated once by `load_context` (same pattern as `company_profile`) | Fetched once, cached |
| `state["messages"]` | `list[BaseMessage]` | Full conversation history — every prior user message AND every prior assistant reply | Appended to by every graph run (LangGraph's `add_messages` reducer) | The literal argument to `graph.stream({"messages": [human_message]}, config, ...)` — LangGraph appends it to the checkpointed list before `analyze_turn` even runs | Grows by exactly one `HumanMessage` (this turn's) + whatever `AIMessage`s previous turns added; the FULL list is sent every time — **no truncation exists** |

**Nothing from the frontend request body reaches the model directly except the recruiter's typed `message` text** — it becomes the newest `HumanMessage` in `state["messages"]`. `session_id` itself is never shown to the model; it's purely a lookup key.

In [3]:
# RUN — this IS the actual prompt-building function, executed live (no network call — pure
# string templating). Demonstrates exactly what the model sees as its SystemMessage.
import sys
sys.path.insert(0, "../backend/..")  # repo root
sys.path.insert(0, "..")

from backend.agent.prompts import build_system_prompt  # EXISTING CODE, imported directly

prompt = build_system_prompt(
    company_profile={"company_name": "Amazon", "industry": "E-commerce & Cloud Computing"},
    job_state={"job_title": "Data Analyst"},
    phase="collecting",
    missing_essential=[],
    jd_exists=False,
    jd_stale=False,
    recruiter_name="Umer",
)
print(prompt[:1200], "\n...\n[truncated —", len(prompt), "chars total]")


You are Arclent, an AI recruiter assistant helping a hiring manager describe a job opening through natural conversation. You are NOT a form — never ask more than one missing question at a time, and never re-ask for information that has already been provided. If asked who/what you are, say you're Arclent.

NEVER bundle two fields into one question. For example, if you still need both experience and employment_type, do NOT ask "What experience level do you want, and should this be full-time, part-time, or internship?" — ask ONLY "How many years of experience should this role require?" first, wait for the reply (or a skip), THEN ask about employment_type on a later turn. This applies everywhere in this prompt that says to ask about a field.

RECRUITER'S NAME: Umer

COMPANY PROFILE (reusable background context — do not repeat it back verbatim unless asked, and never invent facts beyond what is written here):
{
  "company_name": "Amazon",
  "industry": "E-commerce & Cloud Computing"
}

CURR

---
# Part 6 — Model Output

`analyze_turn` forces the model's output into the `TurnAnalysis` schema (`backend/models.py`) via `ChatMistralAI.with_structured_output(TurnAnalysis)`. Every field below is produced on **every** turn — this is one call, not several.

## `TurnAnalysis` — field by field

| Field | Type | Meaning | Example | What backend code does with it | Stored? | Returned to frontend? | Affects next graph step? |
|---|---|---|---|---|---|---|---|
| `intent` | `Intent` enum (11 values) | What the recruiter is doing this turn | `"PROVIDE_INFORMATION"` | Drives branching in `apply_updates` (e.g. blocks field changes for `OFF_TOPIC`/`DOCUMENT_REVIEW`) and `route_after_apply` (routes to `refine_jd` only for `REQUEST_REFINEMENT`) | As `last_intent` in checkpoint | No (not in `ChatResponse`) | Yes — the only field that changes routing |
| `field_updates` | `dict[str,str]` | New/changed scalar `job_state` values | `{"job_title": "Data Analyst"}` | Merged into `job_state` by `apply_field_changes`, restricted to `SCALAR_JOB_FIELDS` | Yes — mirrored into the `jobs` SQL row | Indirectly, via `job_state` | No |
| `list_operations` | `list[ListOperation]` | ADD/REMOVE/REPLACE on skills/responsibilities | `[{"field":"required_skills","operation":"ADD","values":["SQL"]}]` | Applied to the relevant list in `job_state` | Yes — same as above | Indirectly | No |
| `company_overrides` | `dict[str,str]` | Per-job override of a company-profile field | `{"benefits": "This role also gets ESOPs"}` | Merged into `job_state["company_overrides"]` | Yes | Indirectly | No |
| `enough_information` | `bool` | Model's own opinion on sufficiency | `false` | Combined with the deterministic "hard floor" check (never trusted alone — see `sufficiency.py`) to decide `ok` | No (derived `phase` is stored instead) | No | Yes — can flip `phase` to `"summary"` |
| `missing_essential` | `list[str]` | Fields the model thinks are still blocking | `[]` | Merged with the deterministic hard-floor check in `combined_missing_essential` | As `missing_essential` | **Yes** (`ChatResponse.missing_essential`) | Feeds back into next turn's prompt |
| `selected_version` | `"1"\|"2"\|None` | Which JD version the recruiter picked | `null` | Updates `state["selected_version"]` if `"1"`/`"2"` | Yes (`save_selected_version`) | Yes | Gates whether `publish_job` can run |
| `asking_about_field` | `str \| None` | Which ONE optional field this turn's question is about | `"experience"` | Validated against `OPTIONAL_SKIPPABLE_FIELDS` + not-currently-essential (defense in depth — never trusted raw) | As `asking_about_field` | **Yes** | Governs whether the UI can show a "Skip this" button |
| `suggested_options` | `list[str]` | Quick-reply chip labels | `["0-1 years","2-3 years","4-6 years","7+ years"]` | Backfilled/overridden deterministically for known fields (work_mode/employment_type/experience/education) even if the model supplies something else — see `_DEFAULT_OPTIONS_BY_FIELD` | As `suggested_options` | **Yes** | Purely a UI convenience — never changes interpretation of the reply |
| `options_multi_select` | `bool` | Whether several chips can be picked before sending | `true` for skills, `false` for work_mode | Forced `false` whenever the deterministic chip override fires | As `options_multi_select` | **Yes** | UI-only |
| `response` | `str` | The actual chat reply text | `"Got it — Data Analyst. How many years of experience..."` | Markdown-stripped (`_strip_markdown`), appended as an `AIMessage` to `state["messages"]` | As part of the message history (both DBs) | **Yes** (`ChatResponse.assistant_message` + last item of `messages`) | The literal text the recruiter reads |

**Important nuance the schema alone doesn't show**: `apply_updates` and `analyze_turn` apply substantial **deterministic overrides** on top of the raw model output before anything is trusted — e.g. a hard cap (`_SKILLS_FOLLOWUP_CAP = 2`) that forcibly rewrites `response`/`asking_about_field`/`suggested_options` if the model tries to re-ask about skills a third time, and a canonical-chip override for four fixed-choice fields. **The raw LLM output is never returned to the frontend unmodified** — treat `TurnAnalysis` as what the model *proposes*, and the final `ChatResponse` fields as what the backend *decided* after applying these guardrails.

In [4]:
# RUN — the actual JSON schema Mistral is constrained to (this is what
# with_structured_output(TurnAnalysis) compiles into under the hood).
import json
import sys
sys.path.insert(0, "..")
from handoff.models import TurnAnalysis

print(json.dumps(TurnAnalysis.model_json_schema(), indent=2)[:1800], "\n...[truncated]")


{
  "$defs": {
    "Intent": {
      "enum": [
        "PROVIDE_INFORMATION",
        "CORRECT_INFORMATION",
        "FINISH_COLLECTING",
        "REQUEST_JD_GENERATION",
        "SELECT_JD",
        "REQUEST_REFINEMENT",
        "CONFIRM_PUBLISH",
        "CHITCHAT_OR_UNCLEAR",
        "ADVICE_REQUEST",
        "OFF_TOPIC",
        "DOCUMENT_REVIEW"
      ],
      "title": "Intent",
      "type": "string"
    },
    "ListOperation": {
      "properties": {
        "field": {
          "enum": [
            "required_skills",
            "preferred_skills",
            "responsibilities"
          ],
          "title": "Field",
          "type": "string"
        },
        "operation": {
          "enum": [
            "ADD",
            "REMOVE",
            "REPLACE"
          ],
          "title": "Operation",
          "type": "string"
        },
        "values": {
          "items": {
            "type": "string"
          },
          "title": "Values",
          "type": "array"

---
# Part 7 — State Update

`apply_updates` (`backend/agent/nodes.py`) is the ONLY place `TurnAnalysis` output is turned into an updated `job_state`. **EXISTING CODE**, key excerpt:

```python
def apply_updates(state: GraphState, config: RunnableConfig) -> dict:
    analysis = state.get("pending_analysis") or {}
    original_job_state = state.get("job_state") or {}
    intent = analysis.get("intent")

    if intent in _BLOCK_ALL_VALUES:                    # OFF_TOPIC, DOCUMENT_REVIEW
        job_state = apply_field_changes(original_job_state)
    elif intent in _BLOCK_LIST_AND_OVERRIDE_VALUES:     # ADVICE_REQUEST
        job_state = apply_field_changes(original_job_state, analysis.get("field_updates"))
    else:
        job_state = apply_field_changes(
            original_job_state,
            analysis.get("field_updates"),
            analysis.get("list_operations"),
            analysis.get("company_overrides"),
        )

    ok = sufficiency_ok(job_state, llm_enough, llm_missing)
    missing = combined_missing_essential(job_state, llm_missing)
    ...
    if job_state.get("job_title") and company_id is not None:
        upsert_job_draft(session_id, company_id, job_state, jd_stale, owner_user_id=owner_user_id)
```

`apply_field_changes` (shared with the direct `PATCH /job-state` endpoint — see Part 11) is the single place scalar/list/override validation happens:

```python
def apply_field_changes(job_state, field_updates=None, list_operations=None, company_overrides=None):
    job_state = dict(job_state)
    for key, value in (field_updates or {}).items():
        if key in SCALAR_JOB_FIELDS:            # allow-list — unknown keys silently dropped
            job_state[key] = value
    for op in list_operations or []:
        if op.get("field") not in LIST_JOB_FIELDS:
            continue
        job_state[op["field"]] = _apply_list_operation(job_state.get(op["field"], []), op["operation"], op["values"])
    ...
    return job_state
```

**Deterministic sufficiency** (`backend/agent/sufficiency.py`) — never trusts the model's `enough_information` alone:

```python
HARD_REQUIRED_SCALAR = ("job_title",)
HARD_REQUIRED_ANY_OF_LISTS = ("required_skills", "responsibilities")

def hard_floor_met(job_state):
    return all(job_state.get(f) for f in HARD_REQUIRED_SCALAR) and \
           any(job_state.get(f) for f in HARD_REQUIRED_ANY_OF_LISTS)
```

In [5]:
# RUN — apply_field_changes executed live, three turns in a row, showing job_state accumulate.
import sys
sys.path.insert(0, "..")
from backend.agent.nodes import apply_field_changes  # EXISTING CODE, imported directly

state = {}
state = apply_field_changes(state, field_updates={"job_title": "Data Analyst"})
print("After turn 1:", state)

state = apply_field_changes(state, list_operations=[
    {"field": "required_skills", "operation": "ADD", "values": ["SQL", "Python"]}
])
print("After turn 2:", state)

state = apply_field_changes(state, field_updates={"experience": "2-3 years"})
print("After turn 3:", state)

from backend.agent.sufficiency import hard_floor_met
print("hard_floor_met:", hard_floor_met(state))


After turn 1: {'required_skills': [], 'preferred_skills': [], 'responsibilities': [], 'company_overrides': {}, 'job_title': 'Data Analyst'}
After turn 2: {'required_skills': ['SQL', 'Python'], 'preferred_skills': [], 'responsibilities': [], 'company_overrides': {}, 'job_title': 'Data Analyst'}
After turn 3: {'required_skills': ['SQL', 'Python'], 'preferred_skills': [], 'responsibilities': [], 'company_overrides': {}, 'job_title': 'Data Analyst', 'experience': '2-3 years'}
hard_floor_met: True


---
# Part 8 — Database Persistence

Two separate databases, written at different points:

## 1. LangGraph checkpoint (`checkpoints.sqlite`) — authoritative

Written **automatically** by LangGraph itself after every node returns, with no explicit call in application code. Configured in `backend/agent/graph.py`:

```python
_checkpoint_conn = sqlite3.connect(CHECKPOINT_DB_PATH, check_same_thread=False)
checkpointer = SqliteSaver(_checkpoint_conn)
_graph = build_graph().compile(checkpointer=checkpointer)
```

Keyed by `config["configurable"]["thread_id"]` (= `session_id`). Holds the **entire** `GraphState` — message history, `job_state`, `phase`, `jd_versions`, `skills_followup_count`, everything in `handoff/agent_state.py`. This is what makes a conversation resumable across separate HTTP requests: nothing about the conversation is held in server memory between requests.

## 2. `jobs` SQL table (`recruitment.db`) — write-through mirror

Written **explicitly**, inside `apply_updates`, only once `job_title` is known:

```python
if job_state.get("job_title") and company_id is not None:
    upsert_job_draft(session_id, company_id, job_state, jd_stale, owner_user_id=owner_user_id)
```

`upsert_job_draft` (`backend/database.py`) does a plain `INSERT ... ON first call / UPDATE ... on every subsequent call`, keyed by `session_id`. This row is what `list_jobs_for_company`/`list_published_jobs` query — the dashboard's list views never touch LangGraph state at all (see Part 13).

**Why both exist**: LangGraph's checkpoint format isn't something you can easily `SELECT ... WHERE status = 'published'` against — it's a serialized blob per thread. The `jobs` table exists purely so list/detail queries can be plain, fast SQL.

In [6]:
# RUN — full persistence round-trip using the extracted handoff/database.py module, proving
# it's genuinely standalone-executable (not just documentation).
import sys, tempfile, os
sys.path.insert(0, "..")
from handoff import database as db

tmp_path = tempfile.mktemp(suffix=".db")
db.init_db(tmp_path)

company = db.create_company_profile("Acme Inc", {"industry": "Software"}, db_path=tmp_path)
db.create_chat_session("session-abc", user_id=1, company_id=company["id"], db_path=tmp_path)

job_state = {"job_title": "Data Analyst", "required_skills": ["SQL", "Python"], "experience": "2-3 years"}
row = db.upsert_job_draft("session-abc", company["id"], job_state, owner_user_id=1, db_path=tmp_path)
print("jobs row after upsert_job_draft:", {k: row[k] for k in ("job_title", "required_skills", "status", "job_id")})

jd = {"job_title": "Data Analyst", "job_summary": "Great analyst role.", "required_skills": ["SQL", "Python"]}
db.save_jd_versions("session-abc", {"1": jd}, db_path=tmp_path)
db.save_selected_version("session-abc", "1", db_path=tmp_path)

published = db.finalize_publish("session-abc", company["id"], "Acme Inc", jd, "1", db_path=tmp_path)
print("jobs row after finalize_publish:", {k: published[k] for k in ("job_id", "status", "published_at")})

os.remove(tmp_path)


jobs row after upsert_job_draft: {'job_title': 'Data Analyst', 'required_skills': ['SQL', 'Python'], 'status': 'draft', 'job_id': None}
jobs row after finalize_publish: {'job_id': 'AC0001', 'status': 'published', 'published_at': '2026-08-18 07:24:59'}


---
# Part 9 — API Response

`_to_response` (`backend/routes/chat.py`) builds the SAME `ChatResponse` object returned by **every** chat-related endpoint (streamed as the SSE `result` event for `POST /api/chat`, and as plain JSON for the direct-action endpoints):

```python
def _to_response(session_id: str, state: dict) -> ChatResponse:
    job_state = state.get("job_state") or {}
    job_record = get_job_by_session_id(session_id)   # <- fresh SQL read, every single call
    return ChatResponse(
        session_id=session_id,
        phase=state.get("phase", "collecting"),
        assistant_message=state.get("last_response", ""),
        messages=_to_chat_messages(state.get("messages", []), state.get("selected_version")),
        job_state=job_state,
        completeness_pct=_completeness_pct(job_state),
        missing_essential=state.get("missing_essential", []),
        jd_versions=state.get("jd_versions") or None,
        selected_version=state.get("selected_version"),
        jd_stale=state.get("jd_stale", False),
        job_record=job_record,
        asking_about_field=state.get("asking_about_field"),
        suggested_options=state.get("suggested_options") or [],
        options_multi_select=bool(state.get("options_multi_select")),
    )
```

Note it blends **LangGraph state** (everything from `state.get(...)`) with a **live SQL read** (`job_record`) in one response — the frontend never has to reconcile two separate data sources itself.

In [7]:
# RUN — a realistic example ChatResponse, hand-assembled from the real schema to show exact
# shape/types (no live LLM call — see Part 12 for how to hit the real running server).
import json
import sys
sys.path.insert(0, "..")
from handoff.schemas import ChatResponse, ChatMessage

example = ChatResponse(
    session_id="3f1a9c2e-4b1a-4e2a-9c1a-9f6e2b1a2c3d",
    phase="collecting",
    assistant_message="Got it — Data Analyst. How many years of experience should this role require?",
    messages=[
        ChatMessage(role="user", content="Data Analyst"),
        ChatMessage(role="assistant", content="Got it — Data Analyst. How many years of experience should this role require?"),
    ],
    job_state={"job_title": "Data Analyst", "required_skills": [], "preferred_skills": [],
               "responsibilities": [], "company_overrides": {}},
    completeness_pct=8,
    missing_essential=["required_skills_or_responsibilities"],
    jd_versions=None,
    selected_version=None,
    jd_stale=False,
    job_record=None,
    asking_about_field="experience",
    suggested_options=["0-1 years", "2-3 years", "4-6 years", "7+ years"],
    options_multi_select=False,
)
print(json.dumps(example.model_dump(), indent=2))


{
  "session_id": "3f1a9c2e-4b1a-4e2a-9c1a-9f6e2b1a2c3d",
  "phase": "collecting",
  "assistant_message": "Got it \u2014 Data Analyst. How many years of experience should this role require?",
  "messages": [
    {
      "role": "user",
      "content": "Data Analyst",
      "jd_document": null,
      "jd_version": null,
      "is_selected": false
    },
    {
      "role": "assistant",
      "content": "Got it \u2014 Data Analyst. How many years of experience should this role require?",
      "jd_document": null,
      "jd_version": null,
      "is_selected": false
    }
  ],
  "job_state": {
    "job_title": "Data Analyst",
    "required_skills": [],
    "preferred_skills": [],
    "responsibilities": [],
    "company_overrides": {}
  },
  "completeness_pct": 8,
  "missing_essential": [
    "required_skills_or_responsibilities"
  ],
  "jd_versions": null,
  "selected_version": null,
  "jd_stale": false,
  "job_record": null,
  "asking_about_field": "experience",
  "suggested_options":

## SSE framing — how this object actually arrives over the wire

For `POST /api/chat` specifically, the object above is **not** the whole HTTP body — it's the payload of the final SSE frame:

```text
data: {"type": "status", "label": "Understanding your request..."}

data: {"type": "result", "session_id": "3f1a9c2e-...", "phase": "collecting", ...all ChatResponse fields...}

```
(Each frame is `data: <json>` followed by a blank line — the standard SSE frame format. The stream then closes.)

**Why SSE at all?** So the frontend can show a real, honest progress label while the graph runs (`ROUTE_STATUS_LABELS` maps the *actual next node* — predicted via the real `route_after_apply` function, never a separate guess — to a human label like "Creating your job description..."). It is genuinely a stream: multiple frames over one HTTP response, not a single JSON blob. It is **not WebSockets** — there's no bidirectional/persistent channel; each turn is its own fresh POST request. Direct-action endpoints (`PATCH .../job-state`, `POST .../generate`, `POST .../publish`, `POST .../skip-field`) skip SSE entirely and return the same `ChatResponse` as a single ordinary JSON body, because they never need a Mistral call in the loop that would justify a progress indicator (well — `/generate` genuinely does call the model, but synchronously; **cannot determine from the current code** whether this was a deliberate simplicity trade-off or an oversight — worth asking the original author if progress-streaming for `/generate` matters to your UI).

---
# Part 10 — New Frontend Integration (framework-agnostic)

See `handoff/frontend_example.html` for a complete, runnable, dependency-free page. Core SSE-reading logic (vanilla JS, no libraries):

```javascript
async function sendChatMessage(sessionId, message) {
  const response = await fetch("/api/chat", {
    method: "POST",
    headers: { "Content-Type": "application/json" },
    body: JSON.stringify({ session_id: sessionId, message }),
  });
  const reader = response.body.getReader();
  const decoder = new TextDecoder();
  let buffer = "", result = null;

  while (true) {
    const { done, value } = await reader.read();
    if (done) break;
    buffer += decoder.decode(value, { stream: true });
    let boundary;
    while ((boundary = buffer.indexOf("\n\n")) !== -1) {
      const rawEvent = buffer.slice(0, boundary);
      buffer = buffer.slice(boundary + 2);
      const dataLine = rawEvent.split("\n").find(l => l.startsWith("data: "));
      if (!dataLine) continue;
      const event = JSON.parse(dataLine.slice(6));
      if (event.type === "status") { /* show event.label as a progress indicator */ }
      else if (event.type === "result") { delete event.type; result = event; }
    }
  }
  return result;  // the ChatResponse object
}
```

### Equivalent in Python (e.g. for a server-to-server integration, or a non-JS frontend)

In [8]:
# ILLUSTRATIVE — requires a running server + valid session cookie to actually execute.
# Shows the same SSE-parsing logic in Python using `requests` with stream=True.
import json

# `session` is a requests.Session() already carrying the auth cookie
# (e.g. after POSTing to /api/auth/signin).
def send_chat_message(session, base_url, session_id, message):
    resp = session.post(
        f"{base_url}/api/chat",
        json={"session_id": session_id, "message": message},
        stream=True,
    )
    resp.raise_for_status()
    result = None
    buffer = ""
    for chunk in resp.iter_content(chunk_size=None, decode_unicode=True):
        buffer += chunk
        while "\n\n" in buffer:
            raw_event, buffer = buffer.split("\n\n", 1)
            data_line = next((l for l in raw_event.splitlines() if l.startswith("data: ")), None)
            if not data_line:
                continue
            event = json.loads(data_line[len("data: "):])
            if event["type"] == "status":
                print("[status]", event["label"])
            elif event["type"] == "result":
                del event["type"]
                result = event
    return result

print("Function defined — see Part 12 for a version actually run against a live local server.")


Function defined — see Part 12 for a version actually run against a live local server.


---
# Part 11 — Dashboard Rendering (generic, framework-agnostic)

Given one `ChatResponse` object (from ANY of the endpoints — chat turn or direct action), a generic Recruiter Dashboard needs to update three independent regions:

```python
def render_dashboard(data: dict) -> None:
    # 1. Chat transcript — iterate data["messages"]; skip any where m["jd_document"] is truthy
    #    (that's a generated JD rendered as its own card, not a text bubble).
    for m in data["messages"]:
        if m.get("jd_document"):
            continue
        # render_bubble(role=m["role"], text=m["content"])

    # 2. Quick-reply chips — only when non-empty; use options_multi_select to decide
    #    tap-to-send-immediately vs. tap-to-toggle-then-confirm.
    if data["suggested_options"]:
        # render_chips(data["suggested_options"], multi=data["options_multi_select"],
        #              on_pick=lambda opt: send_chat_message(opt))
        pass
    if data["asking_about_field"]:
        # render_skip_button(on_click=lambda: call_direct_endpoint(f"/skip-field"))
        pass

    # 3. Draft panel — every JobState field, plus the generated JD once one exists.
    job_state = data["job_state"]
    # render_field("Job Title", job_state.get("job_title"))
    # render_field("Experience", job_state.get("experience"))
    # render_list_field("Required Skills", job_state.get("required_skills", []))
    # ... (see Part 10's field-by-field table below for the complete set)

    if data["jd_versions"]:
        jd = data["jd_versions"][data["selected_version"]]
        # render_jd_preview(jd)

    # 4. Status bar
    # render_progress(data["completeness_pct"])
    # render_phase_badge(data["phase"])
```

This is intentionally pseudocode — the actual widgets are 100% your choice. The only hard requirement is: **all four regions above come from the exact same response object** on a normal turn; nothing here requires a second network call.

## Recruiter Dashboard Data Contract

The exact fields the dashboard needs, extracted from the real `ChatResponse`/`job_state` shape —
**not** the generic list a template might assume. Every field below is either read straight off
`job_state` (itself sourced from LLM `field_updates`/`list_operations`, confirmed by the recruiter)
or straight off the generated JD (`jd_versions[selected_version]`, LLM-written prose) or is a
value the backend computes deterministically (never LLM-sourced).

| Dashboard field | Origin | API field | Type | Available from | Display guidance |
|---|---|---|---|---|---|
| Job Title | Model output → `job_state` | `job_state.job_title` | `str \| None` | As soon as the recruiter states it (turn 1, typically) | Primary heading of the draft panel |
| Job Category | Model-inferred → `job_state` | `job_state.job_category` | `str \| None` | Same turn as job_title (model infers it, e.g. "Data Analyst" → "Data / Analytics") | Secondary label/tag |
| Experience | Model output → `job_state` | `job_state.experience` | `str \| None` | After the "experience" checklist question is answered or skipped | Free-text display (e.g. "2-3 years") |
| Location | Model output → `job_state` | `job_state.location` | `str \| None` | After the "location" checklist question | A real place name, or "Worldwide" for remote |
| Work Mode | Model output → `job_state` | `job_state.work_mode` | `str \| None` | After the "work_mode" checklist question | One of Remote/Hybrid/Onsite — distinct from Location, never conflate |
| Employment Type | Model output → `job_state` | `job_state.employment_type` | `str \| None` | After the "employment_type" checklist question | Full-time/Part-time/Contract/Internship |
| Education | Model output → `job_state` | `job_state.education` | `str \| None` | Optional, asked after the 4-item checklist | Cosmetic, never a blocker |
| Salary | Model output → `job_state` | `job_state.salary` | `str \| None` | Optional | Free text, whatever format the recruiter gave |
| Deadline | Model output → `job_state` | `job_state.deadline` | `str \| None` | Optional | Free text |
| Required Skills | Model output (list_operations) → `job_state` | `job_state.required_skills` | `list[str]` | Grows incrementally, turn by turn | Tag/chip list, individually removable via `PATCH .../job-state` |
| Preferred Skills | Same | `job_state.preferred_skills` | `list[str]` | Same | Same |
| Responsibilities | Same | `job_state.responsibilities` | `list[str]` | Same | Bullet list |
| Company Overrides | Model output → `job_state` | `job_state.company_overrides` | `dict[str,str]` | Only when the recruiter explicitly asks for THIS job to differ from the company profile | Small "override" indicator per overridden field |
| Job Description (summary) | **LLM generation call**, not the conversational model | `jd_versions[selected_version].job_summary` | `str \| None` | Only after `POST .../generate` is clicked — never before | Main draft-preview text |
| Full JD (all sections) | Same generation call | `jd_versions[selected_version]` (whole object — `about_role`, `major_accountabilities`, `minimum_requirements`, `required_qualifications`, `preferred_qualifications`, `stand_out`, `benefits`, `why_company`, `company_overview`) | nested `dict` | Same — after generation | Full preview / what gets published verbatim |
| Status / Phase | **Backend-computed**, not LLM | `phase` | `str` enum (6 values) | Every response | Drives which buttons are enabled (e.g. Publish only valid once `jd_versions` exist and `jd_stale` is false) |
| Completeness | **Backend-computed** (`_completeness_pct`) | `completeness_pct` | `int` 0-100 | Every response | Progress bar — UX-only, never gates any action |
| Suggested Options (chips) | Model output, deterministically overridden for known fields | `suggested_options` | `list[str]` | Only on a turn that's a question | Tappable quick-replies |
| Multi-select flag | Model output | `options_multi_select` | `bool` | Alongside `suggested_options` | Governs tap-once-send vs. tap-several-then-confirm |
| Skip-eligible field | Backend-validated (never raw model output) | `asking_about_field` | `str \| None` | Only when the current question is about a genuinely optional field | Show a "Skip this" button when non-null |
| Conversation Messages | Full turn history | `messages[]` (each `{role, content, jd_document, jd_version, is_selected}`) | `list[ChatMessage]` | Every response (full history, not just the delta) | Chat transcript — skip any entry where `jd_document` is truthy when rendering as text (render as a JD card instead) |
| Job Requisition ID | **Backend-assigned at publish time**, not LLM | `job_record.job_id` | `str \| None` | Only after `POST .../publish` succeeds | e.g. "AM0001" — null before publish |
| Accepting Applications | Backend/DB | `job_record.accepting_applications` | `bool` (as `0`/`1` in SQL) | Only meaningful once published | Toggle, via the separate `PUT /api/jobs/{session_id}/accepting-applications` endpoint (not part of the chat contract) |

---
# Part 12 — Reopening an Existing Job

## `GET /api/chat/{session_id}`

**EXISTING CODE** (`backend/routes/chat.py`):

```python
@router.get("/{session_id}", response_model=ChatResponse)
def get_chat(session_id: str, user=Depends(get_current_recruiter)) -> ChatResponse:
    _authorize_session(session_id, user)
    graph = get_compiled_graph()
    snapshot = graph.get_state({"configurable": {"thread_id": session_id}})

    if snapshot.values.get("messages"):
        return _to_response(session_id, snapshot.values)     # (A) normal case

    # (B) No LangGraph checkpoint exists for this thread at all (e.g. a job that was
    # seeded directly into the DB, never run through the graph). If it's published,
    # synthesize a read-only view straight from the `jobs` row — no graph run, no LLM call.
    record = get_job_by_session_id(session_id)
    if record and record.get("status") == "published":
        job_state = _job_state_from_record(record)
        greeting = f"This job is already published as {record.get('job_id')}. What would you like to change?"
        synthetic_state = {"job_state": job_state, "phase": "published", ...,
                            "messages": [AIMessage(content=greeting)]}
        return _to_response(session_id, synthetic_state)

    raise HTTPException(status_code=404, detail="No conversation found for this session_id")
```

**Response shape is identical to a chat turn's** — the same `ChatResponse`. A new frontend calls this exactly once, when the modal/page opens (first load, or "Manage → Edit" on a job in the dashboard list), NOT on every render.

## During an active chat turn (contrast)

The Draft panel receives its data **directly from the `/api/chat` SSE result** — no separate GET happens mid-conversation. `GET /{session_id}` is only for the *reopening* moment.

In [9]:
# ILLUSTRATIVE — requires a running local server (`uvicorn backend.main:app`) and a signed-in
# session to actually execute. Demonstrates the reopen flow end-to-end against the real API.
CODE = '''
import requests

BASE = "http://127.0.0.1:8000"
s = requests.Session()
s.post(f"{BASE}/api/auth/signin", json={"email": "you@example.com", "password": "..."})

# First turn of a brand-new job:
r = s.post(f"{BASE}/api/chat", json={"session_id": None, "message": "Data Analyst"}, stream=True)
# ...parse SSE as shown in Part 10, extract session_id from the "result" event...

# Later, reopening the SAME job (e.g. after navigating away and back):
r2 = s.get(f"{BASE}/api/chat/{session_id}")
resumed = r2.json()   # plain JSON here — GET is NOT streamed, unlike POST /api/chat
print(resumed["phase"], resumed["job_state"])
'''
print(CODE)



import requests

BASE = "http://127.0.0.1:8000"
s = requests.Session()
s.post(f"{BASE}/api/auth/signin", json={"email": "you@example.com", "password": "..."})

# First turn of a brand-new job:
r = s.post(f"{BASE}/api/chat", json={"session_id": None, "message": "Data Analyst"}, stream=True)
# ...parse SSE as shown in Part 10, extract session_id from the "result" event...

# Later, reopening the SAME job (e.g. after navigating away and back):
r2 = s.get(f"{BASE}/api/chat/{session_id}")
resumed = r2.json()   # plain JSON here — GET is NOT streamed, unlike POST /api/chat
print(resumed["phase"], resumed["job_state"])



---
# Part 13 — Job List Flow (separate from everything above)

This is a **completely independent** flow — no LangGraph, no LLM, no `session_id`/thread lookup. Plain SQL against the `jobs` table's write-through mirror.

## Recruiter's own dashboard — `GET /api/jobs`

```python
@router.get("")
def get_jobs_for_recruiter(user=Depends(get_current_recruiter)) -> list[dict]:
    return list_jobs_for_company(user["company_id"])
```
Returns every draft + published job belonging to the recruiter's own company (`jobs.company_id = ?`), most-recently-updated first. This is what a "Recent Jobs" list on the dashboard should call **on page load**, independent of any chat session.

## Public job board — `GET /api/public/jobs` and `GET /api/public/jobs/{job_id}`

```python
@public_router.get("")
def get_public_jobs() -> list[dict]:
    return list_published_jobs()          # published jobs, ALL companies, no auth required

@public_router.get("/{job_id}")
def get_public_job(job_id: str) -> dict:
    job = get_published_job_by_job_id(job_id)
    if not job:
        raise HTTPException(status_code=404, detail="Job not found")
    return job                             # full row incl. selected_jd — the actual JD content
```

## CSV export — `GET /api/jobs/report`

Real fields only (title/status/location/employment_type/dates/accepting_applications) — **cannot determine an applications/proposals table in the current schema**, so no funnel/hire metrics exist to export.

## Field shape returned by the list endpoints

`_JOB_LIST_COLUMNS` (in `handoff/database.py`) is a deliberately narrower column set than the full `jobs` row — no `jd_version_1`/`selected_jd` (the full JD text) is included in list views, only in `get_published_job_by_job_id`'s detail query and in a chat session's own `ChatResponse.job_record`.

In [10]:
# RUN — list_jobs_for_company / list_published_jobs executed live against the same temp DB
# used in Part 8, proving this really is independent SQL (no LangGraph object touched at all).
import sys, tempfile, os
sys.path.insert(0, "..")
from handoff import database as db

tmp_path = tempfile.mktemp(suffix=".db")
db.init_db(tmp_path)
company = db.create_company_profile("Acme Inc", {}, db_path=tmp_path)
db.upsert_job_draft("s1", company["id"], {"job_title": "Data Analyst"}, db_path=tmp_path)
db.upsert_job_draft("s2", company["id"], {"job_title": "Backend Engineer"}, db_path=tmp_path)

for j in db.list_jobs_for_company(company["id"], db_path=tmp_path):
    print(j["job_title"], "-", j["status"])

os.remove(tmp_path)


Data Analyst - draft
Backend Engineer - draft


---
# Part 14 — Complete Data Lifecycle: "Data Analyst"

```text
Recruiter types:  "Data Analyst"
   │
   ▼
New Frontend
   │  builds { "session_id": null, "message": "Data Analyst" }
   ▼
API Request
   │  POST /api/chat, Content-Type: application/json, body = above JSON
   ▼
Backend API  (backend/routes/chat.py :: post_chat)
   │  session_id is null -> mints a UUID, INSERTs a chat_sessions row (ownership record)
   ▼
Request Validation
   │  FastAPI parses the body into a ChatRequest Pydantic model — a non-conforming body
   │  (e.g. missing "message") is rejected with 422 before any of this even runs
   ▼
LangGraph State
   │  graph.stream({"messages": [HumanMessage("Data Analyst")]}, config) —
   │  LangGraph's `add_messages` reducer appends this to the (empty, turn-1) message list
   ▼
load_context node
   │  company_id/user_id present -> hydrates company_profile + recruiter_name from SQL
   ▼
analyze_turn node  ──▶  AI / LLM  (Mistral, call_structured(TurnAnalysis, messages))
   │  in:  SystemMessage(company_profile + job_state={} + phase="collecting" + ...) + [HumanMessage("Data Analyst")]
   │  out: TurnAnalysis(intent=PROVIDE_INFORMATION, field_updates={"job_title":"Data Analyst"},
   │                     response="Got it — Data Analyst. How many years of experience...?",
   │                     asking_about_field="experience", suggested_options=["0-1 years",...])
   ▼
Structured Model Output
   │  format: a validated TurnAnalysis Pydantic instance (guaranteed schema-conformant)
   ▼
apply_updates node  (business logic)
   │  apply_field_changes({}, field_updates={"job_title":"Data Analyst"}) -> job_state={"job_title":"Data Analyst",...}
   │  hard_floor_met? NO (no skills/responsibilities yet) -> phase stays "collecting"
   │  job_state.get("job_title") is truthy -> upsert_job_draft(...) fires (SQL write, see below)
   ▼
Updated job_state
   │  {"job_title": "Data Analyst", "required_skills": [], "preferred_skills": [],
   │   "responsibilities": [], "company_overrides": {}}
   ▼
Database / LangGraph checkpoint
   │  (1) LangGraph SqliteSaver auto-persists the full GraphState to checkpoints.sqlite
   │  (2) upsert_job_draft() INSERTs a new `jobs` row into recruitment.db:
   │      job_title="Data Analyst", session_id=<the new uuid>, company_id=<recruiter's company>,
   │      status="draft", required_skills="[]", ...
   ▼
Backend Response
   │  _to_response() builds ChatResponse — job_state as above, phase="collecting",
   │  asking_about_field="experience", suggested_options=["0-1 years","2-3 years","4-6 years","7+ years"],
   │  job_record=<the fresh SQL row just inserted>
   │  yielded as SSE:  data: {"type":"result", ...all fields...}
   ▼
New Frontend
   │  parses the SSE stream, captures session_id for all future calls, gets the ChatResponse object
   ▼
Recruiter Dashboard / Draft
      renders: chat bubble "Got it — Data Analyst. How many years..."
               chips ["0-1 years","2-3 years","4-6 years","7+ years"] + a Skip button
               draft panel: Job Title = "Data Analyst"
               completeness bar: (job_title now counts toward completeness_pct)
```

---
# Part 15 — Developer Integration Checklist

## A. EXISTING SYSTEM — preserve as-is, do not modify unless you have a specific reason

- [ ] Mistral as the LLM (`ChatMistralAI`, `mistral-large-latest`), invoked via `.with_structured_output(...)`
- [ ] LangGraph as the orchestration framework — the `load_context → analyze_turn → apply_updates → {...}` graph shape
- [ ] The `TurnAnalysis` / `JobDescriptionDraft` / `JDRefinementOutput` structured-output schemas (`backend/models.py`)
- [ ] `SYSTEM_PROMPT_TEMPLATE` and the two JD prompt templates (`backend/agent/prompts.py`) — this IS the product behavior
- [ ] The deterministic guardrails layered on top of raw LLM output (hard floor, skills-loop cap, canonical chip overrides, skip-field tracking) — these fix real, previously-shipped bugs; removing them reintroduces them
- [ ] SSE as the transport for `POST /api/chat` — a real streamed progress indicator, not decorative
- [ ] The "direct action" pattern: generation and publishing are NEVER triggered by chat messages, only by their own dedicated endpoints

## B. API CONTRACT — what your new frontend must send/expect for the existing backend to keep working unmodified

- [ ] `POST /api/chat` — body `{session_id, message}`, response is SSE with `status`*/`result` frames
- [ ] `GET /api/chat/{session_id}` — plain JSON, for reopening only
- [ ] `PATCH /api/chat/{session_id}/job-state` — silent field edits, plain JSON, body = `JobStatePatch`
- [ ] `POST /api/chat/{session_id}/generate` — no body, plain JSON response; 400 if hard floor isn't met
- [ ] `POST /api/chat/{session_id}/skip-field` — no body; acts on whatever `asking_about_field` currently is
- [ ] `POST /api/chat/{session_id}/publish` — no body; 400 if no JD exists or it's stale
- [ ] `GET /api/jobs`, `GET /api/public/jobs`, `GET /api/public/jobs/{job_id}` — independent list/detail flow
- [ ] Every one of the above (except the two public ones) requires the caller to be authenticated — see section C

## C. INTEGRATION WORK — what you specifically need to build/adapt for the larger platform

- [ ] **Auth**: swap `get_current_recruiter`/the cookie-session mechanism for your platform's own auth, as long as it yields something with `id` (int) and `company_id` (int) — every function downstream only needs those two
- [ ] **`company_id` provisioning**: `company_profile` rows must exist before jobs can be created; decide whether your platform's existing "company"/"organization" table replaces this table entirely (in which case adapt `handoff/database.py`'s FK) or coexists with it
- [ ] **Database choice**: the existing system is SQLite-via-raw-`sqlite3`, no ORM, no connection pool — if your platform standardizes on Postgres/MySQL or an ORM, `handoff/database.py` needs real porting (column types, `AUTOINCREMENT`→`SERIAL`, JSON-as-TEXT→native JSON columns, `BEGIN IMMEDIATE` locking semantics)
- [ ] **LangGraph checkpointer**: currently `SqliteSaver` against a local file — if you're running multiple backend instances, this needs to move to a shared backend (LangGraph supports Postgres checkpointers) or conversations will only be resumable on the instance that started them
- [ ] **Environment/config**: `MISTRAL_API_KEY`, `MISTRAL_MODEL`, `APP_DB_PATH`, `CHECKPOINT_DB_PATH` — all read via `os.getenv` in `backend/config.py`; wire these into your platform's own config/secrets system
- [ ] **Frontend**: entirely your choice — `handoff/frontend_example.html` proves the contract is self-describing and framework-agnostic; build whatever UI fits the larger platform
- [ ] **Rate limiting**: the existing `call_structured` has a short backoff-and-retry for Mistral 429s (`backend/agent/llm.py`) — if you expect materially higher traffic than this app currently sees, revisit the retry/backoff constants and consider a request queue

## Explicit gaps — **cannot determine from the current code**

- No applications/candidate-tracking table exists — "Total Proposals"/hire-funnel metrics are not implementable from this schema as-is.
- No multi-process/multi-instance deployment story is implemented (single SQLite file, in-process LangGraph graph singleton) — horizontal scaling needs the LangGraph checkpointer + `APP_DB_PATH` changes noted above.
- No automated test suite was found for this component during inspection — verify behavior via the live endpoints (Part 12's pattern) before relying on any specific edge case.